In [1]:
import zipfile
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import optuna # <-- NUEVO: Importar optuna
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

2025-06-17 18:46:03.120601: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750196763.604661   42537 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750196763.714508   42537 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750196764.689461   42537 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750196764.689506   42537 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750196764.689508   42537 computation_placer.cc:177] computation placer alr

In [2]:
zip_file_path = '../data/MeIA2025-Reto-01.zip'

extracted_folder_path = '../data/extracted_corpus/'

# 2. Crear la carpeta de extracción si no existe
if not os.path.exists(extracted_folder_path):
    os.makedirs(extracted_folder_path)
    print(f"Carpeta '{extracted_folder_path}' creada.")
# 3. Descomprimir archivo  
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_folder_path)
    print(f"'{zip_file_path}' descomprimido exitosamente en '{extracted_folder_path}'.")
except FileNotFoundError:
    print(f"Error: El archivo ZIP no se encontró en '{zip_file_path}'. Verifica la ruta.")
except Exception as e:
    print(f"Ocurrió un error al descomprimir el archivo: {e}")
    
# 4. Listar los archivos descomprimidos (para verificar)
print("\nArchivos en la carpeta del corpus:")
corpus_files = os.listdir(extracted_folder_path)
for file_name in corpus_files:
    print(f"- {file_name}")


'../data/MeIA2025-Reto-01.zip' descomprimido exitosamente en '../data/extracted_corpus/'.

Archivos en la carpeta del corpus:
- Datos-MeIA-Reto-01


In [3]:
# Definir la ruta base donde se extrajo el contenido del ZIP
# Asegúrate de que esta ruta sea correcta relativa a tu notebook test.ipynb
# Si tu notebook está en 'notebooks/' y la extracción está en 'data/extracted_corpus/Datos-MelA-Reto-01/'
base_extracted_path = '../data/extracted_corpus/Datos-MeIA-Reto-01/'

# Rutas completas a los archivos XLSX
train_file_path = os.path.join(base_extracted_path, 'MeIA_2025_train.xlsx')
test_file_path = os.path.join(base_extracted_path, 'MeIA_2025_test_wo_labels.xlsx')

print(f"Intentando cargar el archivo de entrenamiento desde: {train_file_path}")
print(f"Intentando cargar el archivo de prueba desde: {test_file_path}")

try:
    # Cargar el dataset de entrenamiento
    
    df_train = pd.read_excel(train_file_path)
    print(f"Datos de entrenamiento cargados correctamente")
    # Cargar el dataset de prueba (sin etiquetas)
    df_test = pd.read_excel(test_file_path)
    print(f"Datos de test cargados correctamente")


except FileNotFoundError:
    print(f"Error: Uno de los archivos XLSX no se encontró.")
    print(f"Asegúrate de que las rutas sean correctas: '{train_file_path}' y '{test_file_path}'")
    print(f"Y que la carpeta 'Datos-MelA-Reto-01' esté dentro de 'extracted_corpus'.")
except Exception as e:
    print(f"Ocurrió un error al cargar los archivos Excel: {e}")


Intentando cargar el archivo de entrenamiento desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_train.xlsx
Intentando cargar el archivo de prueba desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_test_wo_labels.xlsx
Datos de entrenamiento cargados correctamente
Datos de test cargados correctamente


### Viejo

In [7]:
# Preparar el DataFrame
df_train_roberta = df_train.rename(columns={'structured_text': 'text', 'Polarity': 'label'})
df_train_roberta['label'] = df_train_roberta['label'].apply(lambda x: int(x) - 1)

dataset = Dataset.from_pandas(df_train_roberta)
train_test_split = dataset.train_test_split(test_size=0.1, seed=42) # Añadido seed para reproducibilidad
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

# --- Cargar el tokenizador y el modelo para FacebookAI/xlm-roberta-base ---
model_name = "pysentimiento/robertuito-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5) # Asegúrate de que num_labels sea 5 para tus 5 clases de polaridad

# Función de tokenización
# Los parámetros de padding, truncation y max_length son generalmente adecuados para XLM-RoBERTa
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256 # Puedes ajustar este valor si tus textos son más largos o más cortos
    )

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Función para calcular métricas (se mantiene igual)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./results_xlm_roberta", # Directorio de salida diferente para este modelo
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs_xlm_roberta', # Directorio de logs diferente
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    # report_to="none" # Descomenta si no quieres reportar a Weights & Biases u otros
)

# Inicializar el Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# Iniciar el afinamiento
trainer.train()

print("¡Afinamiento del modelo FacebookAI/xlm-roberta-base completado!")

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

RuntimeError: Error(s) in loading state_dict for Linear:
	size mismatch for bias: copying a param with shape torch.Size([3]) from checkpoint, the shape in current model is torch.Size([5]).

In [5]:
# Preparar el DataFrame
df_train_roberta = df_train.rename(columns={'structured_text': 'text', 'Polarity': 'label'})
df_train_roberta['label'] = df_train_roberta['label'].apply(lambda x: int(x) - 1)

dataset = Dataset.from_pandas(df_train_roberta)
train_test_split = dataset.train_test_split(test_size=0.1, seed=42) # Añadido seed para reproducibilidad
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

# --- Cargar el tokenizador y el modelo para FacebookAI/xlm-roberta-base ---
model_name = "FacebookAI/xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5) # Asegúrate de que num_labels sea 5 para tus 5 clases de polaridad

# Función de tokenización
# Los parámetros de padding, truncation y max_length son generalmente adecuados para XLM-RoBERTa
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256 # Puedes ajustar este valor si tus textos son más largos o más cortos
    )

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Función para calcular métricas (se mantiene igual)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./results_xlm_roberta", # Directorio de salida diferente para este modelo
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs_xlm_roberta', # Directorio de logs diferente
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    # report_to="none" # Descomenta si no quieres reportar a Weights & Biases u otros
)

# Inicializar el Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# Iniciar el afinamiento
trainer.train()

print("¡Afinamiento del modelo FacebookAI/xlm-roberta-base completado!")

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.104600,1.151079,0.516000,0.495192
2,0.950800,1.057804,0.528000,0.521796
3,0.866300,1.052354,0.548000,0.545240


¡Afinamiento del modelo FacebookAI/xlm-roberta-base completado!


In [6]:
# Ruta a la carpeta donde quieres guardar el modelo
output_dir = "./modelos/xml-roberta"

# Guardar el modelo y el tokenizador en esa carpeta
trainer.save_model(output_dir)

In [4]:
# --- Asumimos que tu DataFrame 'df_train' ya está cargado ---

# Preparar el DataFrame
df_train_tabularisai = df_train.rename(columns={'Review': 'text', 'Polarity': 'label'})
df_train_tabularisai['label'] = df_train_tabularisai['label'].apply(lambda x: int(x) - 1)

dataset = Dataset.from_pandas(df_train_tabularisai)
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

# Cargar el tokenizador y el modelo que seleccionaste
model_name = "tabularisai/multilingual-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

training_args = TrainingArguments(
    output_dir="./results_tabularisai",
    num_train_epochs=3,
    learning_rate=2e-5, # Un learning rate bajo es bueno para el fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs_tabularisai',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# Iniciar el afinamiento
trainer.train()

print("¡Afinamiento del modelo de tabularisai completado!")

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.025400,1.087794,0.534000,0.537266
2,0.881900,1.111234,0.528000,0.528628
3,0.771000,1.154419,0.528000,0.530826


¡Afinamiento del modelo de tabularisai completado!


In [5]:
# Ruta a la carpeta donde quieres guardar el modelo
output_dir = "./modelos/tabularisai"

# Guardar el modelo y el tokenizador en esa carpeta
trainer.save_model(output_dir)

### bert-base-multilingual-uncased-sentiment 

In [4]:
# Preparar el DataFrame
df_train_nlptown = df_train.rename(columns={'Review': 'text', 'Polarity': 'label'})
df_train_nlptown['label'] = df_train_nlptown['label'].apply(lambda x: int(x) - 1) # Ajustamos a 0-4

# Convertir a Dataset y dividir
dataset = Dataset.from_pandas(df_train_nlptown)
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']


# Cargar el tokenizador y el modelo que seleccionaste
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
# Las etiquetas en este modelo son '1 star' a '5 stars', pero el Trainer las manejará internamente
# si le pasamos num_labels=5 y las etiquetas de 0 a 4.
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

training_args = TrainingArguments(
    output_dir="./results_nlptown",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5, # Un learning rate más bajo suele ser bueno para el fine-tuning
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs_nlptown',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# Iniciar el afinamiento
trainer.train()

print("¡Afinamiento del modelo de nlptown completado!")

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.048400,1.104344,0.484000,0.477437
2,0.816600,1.077669,0.528000,0.524725
3,0.664300,1.195405,0.544000,0.543270


¡Afinamiento del modelo de nlptown completado!


In [5]:
# Ruta a la carpeta donde quieres guardar el modelo
output_dir = "./modelos/bert-sentiment"

# Guardar el modelo y el tokenizador en esa carpeta
trainer.save_model(output_dir)

### dccuchile/bert-base-spanish-wwm-cased

In [4]:
print("Creando input estructurado...")
df_train['structured_text'] = df_train.apply(
    lambda row: f"tipo: {str(row['Type']).lower()}. pueblo: {str(row['Town']).lower()}. reseña: {str(row['Review']).lower()}",
    axis=1
)


Creando input estructurado...


In [17]:
# Paso 2: Preparar el DataFrame y convertirlo a un Dataset de Hugging Face
# Renombramos las columnas y ajustamos las etiquetas (si no lo has hecho en el df original)
df_train_beto = df_train.rename(columns={'structured_text': 'text', 'Polarity': 'label'})
df_train_beto['label'] = df_train_beto['label'].apply(lambda x: int(x) - 1)

# Convertir a Dataset
dataset = Dataset.from_pandas(df_train_beto)

# Dividir en entrenamiento y validación
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']


# Paso 3: Cargar el tokenizador y el modelo BETO
# ¡Este es el cambio principal!
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5) # 5 clases de polaridad

# Paso 4: Tokenizar los datos
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Paso 5: Definir la métrica de evaluación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

# Paso 6: Configurar y ejecutar el entrenamiento
training_args = TrainingArguments(
    output_dir="./results_beto",     # Nuevo directorio para no sobreescribir el anterior
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_beto',       # Nuevo directorio de logs
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# ¡Iniciar el entrenamiento!
trainer.train()

print("¡Entrenamiento con BETO completado!")

# Para guardar el modelo final y el tokenizador
# trainer.save_model("./mi_modelo_beto_final")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.118000,1.042671,0.520000,0.478475
2,1.039900,1.035133,0.518000,0.456780
3,0.536400,1.021053,0.568000,0.570595


¡Entrenamiento con BETO completado!


In [18]:
# Ruta a la carpeta donde quieres guardar el modelo
output_dir = "./modelos/bert"

# Guardar el modelo y el tokenizador en esa carpeta
trainer.save_model(output_dir)